# MEF2 Isoform Downloader
UniProt API를 이용해 human MEF2A/B/C/D의 모든 isoform 서열을 받아 종/유전자별로 저장합니다.

**저장 구조**
```
MEF2/Protein-sequences/Homo-sapiens/
    Homo-sapiens-MEF2A-I1.fasta
    Homo-sapiens-MEF2A-I2.fasta
    Homo-sapiens-MEF2A-all.fasta
    ...
```


In [2]:
import os
import requests
import time

## 설정
Accession 추가/제거로 대상 유전자를 바꿀 수 있어요.

In [6]:
# Human MEF2 계열 UniProt accession
MEF2_ACCESSIONS = {
    "MEF2A": "Q02078",
    "MEF2B": "Q02080",
    "MEF2C": "Q06413",
    "MEF2D": "Q14814",
}

OUTPUT_BASE = "/rna/liha/phylogenomics_practice/MEF2/Protein-sequences/Homo-sapiens"
os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f"저장 폴더 준비 완료: {OUTPUT_BASE}")


저장 폴더 준비 완료: /rna/liha/phylogenomics_practice/MEF2/Protein-sequences/Homo-sapiens


## 함수 정의

In [7]:
def fetch_isoforms(gene_name, accession):
    """UniProt REST API로 isoform 서열 전부 가져오기"""
    url = (
        "https://rest.uniprot.org/uniprotkb/stream"
        "?format=fasta"
        f"&query=accession:{accession}"
        "&includeIsoform=true"
    )
    
    print(f"[{gene_name}] 요청 중... ({url})")
    
    response = requests.get(url)
    if response.status_code != 200:
        print(f"  오류: HTTP {response.status_code}")
        return []
    
    raw = response.text.strip()
    
    # FASTA 레코드 분리
    records = []
    current = []
    for line in raw.splitlines():
        if line.startswith(">") and current:
            records.append("\n".join(current))
            current = []
        current.append(line)
    if current:
        records.append("\n".join(current))
    
    return records


def parse_isoform_number(header):
    """헤더에서 isoform 번호 파싱
    예) >sp|Q02078-2|MEF2A_HUMAN -> I2
        >sp|Q02078|MEF2A_HUMAN   -> I1 (canonical)
    """
    accession_field = header.split("|")[1]
    if "-" in accession_field:
        iso_num = accession_field.split("-")[1]
        return f"I{iso_num}"
    else:
        return "I1"


def save_isoforms(gene_name, records):
    saved = []
    for record in records:
        lines = record.splitlines()
        header = lines[0]
        iso_label = parse_isoform_number(header)
        
        # 파일명 변경
        filename = f"MEF2{gene_name[-1]}-{iso_label}-Homo-sapiens.fasta"
        filepath = os.path.join(OUTPUT_BASE, filename)
        
        # 헤더 변경
        new_header = f">MEF2{gene_name[-1]}-{iso_label}-Homo-sapiens"
        new_record = "\n".join([new_header] + lines[1:])
        
        with open(filepath, "w") as f:
            f.write(new_record + "\n")
        
        saved.append(filename)
        print(f"  저장: {filename}")
    
    return saved


def merge_all(gene_name, records):
    """모든 isoform을 하나의 파일로도 저장"""
    merged_path = os.path.join(OUTPUT_BASE, f"Homo-sapiens-{gene_name}-all.fasta")
    with open(merged_path, "w") as f:
        for record in records:
            f.write(record + "\n\n")
    print(f"  병합 저장: Homo-sapiens-{gene_name}-all.fasta")


print("함수 정의 완료")


함수 정의 완료


## 다운로드 실행

In [8]:
summary = {}

for gene_name, accession in MEF2_ACCESSIONS.items():
    print(f"{'='*40}")
    records = fetch_isoforms(gene_name, accession)
    
    if not records:
        print(f"  {gene_name}: 서열을 가져오지 못했습니다.\n")
        continue
    
    print(f"  {len(records)}개 isoform 발견")
    saved = save_isoforms(gene_name, records)
    merge_all(gene_name, records)
    summary[gene_name] = saved
    
    time.sleep(0.5)  # UniProt API rate limit 대비
    print()

print("=" * 40)
print("다운로드 완료!")


[MEF2A] 요청 중... (https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=accession:Q02078&includeIsoform=true)
  8개 isoform 발견
  저장: MEF2A-I1-Homo-sapiens.fasta
  저장: MEF2A-I5-Homo-sapiens.fasta
  저장: MEF2A-I2-Homo-sapiens.fasta
  저장: MEF2A-I3-Homo-sapiens.fasta
  저장: MEF2A-I4-Homo-sapiens.fasta
  저장: MEF2A-I6-Homo-sapiens.fasta
  저장: MEF2A-I7-Homo-sapiens.fasta
  저장: MEF2A-I8-Homo-sapiens.fasta
  병합 저장: Homo-sapiens-MEF2A-all.fasta

[MEF2B] 요청 중... (https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=accession:Q02080&includeIsoform=true)
  2개 isoform 발견
  저장: MEF2B-I1-Homo-sapiens.fasta
  저장: MEF2B-I2-Homo-sapiens.fasta
  병합 저장: Homo-sapiens-MEF2B-all.fasta

[MEF2C] 요청 중... (https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=accession:Q06413&includeIsoform=true)
  6개 isoform 발견
  저장: MEF2C-I1-Homo-sapiens.fasta
  저장: MEF2C-I2-Homo-sapiens.fasta
  저장: MEF2C-I3-Homo-sapiens.fasta
  저장: MEF2C-I4-Homo-sapiens.fasta
  저장: MEF2C-I5-Homo-sapiens.fasta
  저장: MEF2C-I6-H

In [9]:
import glob

output_path = os.path.join(OUTPUT_BASE, "Homo-sapiens-MEF.fasta")

with open(output_path, "w") as outfile:
    for gene_name in MEF2_ACCESSIONS:
        pattern = os.path.join(OUTPUT_BASE, f"MEF2{gene_name[-1]}-*-Homo-sapiens.fasta")
        for filepath in sorted(glob.glob(pattern)):
            with open(filepath) as infile:
                outfile.write(infile.read())

print(f"저장 완료: {output_path}")

# 몇 개 들어갔는지 확인
with open(output_path) as f:
    count = sum(1 for l in f if l.startswith(">"))
print(f"총 {count}개 서열 포함")

저장 완료: /rna/liha/phylogenomics_practice/MEF2/Protein-sequences/Homo-sapiens/Homo-sapiens-MEF.fasta
총 22개 서열 포함
